# ETL — V1DD 1196 synapses (`SynapseConnectivityLong` + `SynapseFeatureMatrix`)

Registers the full V1DD `1196` synapse table into the two synapse classes and
writes them to the shared V1DD output root, alongside the other `etl_v1dd_*`
notebooks.

- **`synapse/`** — long `SynapseConnectivityLong` table, one row per synapse
  (`id`, `presynaptic_cell`, `postsynaptic_cell`, `dataset_id`, `project_id`).
  The pre/post pair is intentionally *not* unique.
- **`synapsefeatures/<feature_matrix_id>/`** — wide per-synapse feature table
  (position, `size`, `synaptictargetlabel`), built from raw dataframes exactly
  like `cellfeatures/` and LEFT-joined on the synapse `id` at read time.
- **`synapsefeaturematrix/`** — one `SynapseFeatureMatrix` pointer row locating
  the wide table.

At ~8M rows the long table is written straight from a dataframe with
`write_deltalake` (building one pydantic model per synapse would be needlessly
heavy); the small `SynapseFeatureMatrix` pointer row goes through `write_models`.
Both land on exactly the layout the registered `WriteSpec` entries describe, so
`read_synapse_table` reads them back.

In [1]:
from pathlib import Path

import pandas as pd
import pyarrow as pa
from deltalake import write_deltalake

from connects_common_connectivity.models import SynapseConnectivityLong, SynapseFeatureMatrix
from connects_common_connectivity.io import read_synapse_table, write_models
from connects_common_connectivity.io.arrow_utils import attach_linkml_metadata, build_arrow_schema

In [2]:
# --- Constants -------------------------------------------------------------
PROJECT_ID = "v1dd"
DATASET_ID = "v1dd_1196_em"
FEATURE_MATRIX_ID = "v1dd_1196_synapse_features"

OUTPUT_ROOT = Path("../results/v1dd_1196_v3/")

## Load source

`syn_df` is one row per synapse (`id` unique); `syn_label_df` gives a
spine/shaft/soma `tag` for a **subset** of synapses (renamed to
`synaptictargetlabel`). Root ids are 18-digit ints kept as strings — never
cast.

In [3]:
DATA_ROOT = Path("/data/v1dd_1196")
syn_df = pd.read_feather(DATA_ROOT / "syn_df_all_to_proofread_to_all_1196.feather")
for col in ("id", "pre_pt_root_id", "post_pt_root_id"):
    syn_df[col] = syn_df[col].astype(str)

label_df = (
    pd.read_feather(DATA_ROOT / "syn_label_df_all_to_proofread_to_all_1196.feather")
    .reset_index()                                  # 'id' is the feather index
    .rename(columns={"tag": "synaptictargetlabel"})
    .astype({"id": str})
)
print("syn_df:", syn_df.shape,
      "| labels cover", f"{label_df['id'].nunique()/len(syn_df):.1%}", "of synapses")

syn_df: (8204497, 13) | labels cover 81.7% of synapses


## Write 1 — long single-synapse table → `synapse/`

Long-form `SynapseConnectivityLong` columns, built straight from the dataframe
into an Arrow table (with the model's schema + LinkML metadata), then written
with a `(project_id, dataset_id)`-scoped overwrite. `write_models(long_rows, ...)`
is the equivalent one-liner and is the right call at smaller scale, but here it
would mean instantiating ~8M pydantic models.

In [4]:
long_df = pd.DataFrame({
    "id": syn_df["id"],
    "presynaptic_cell": syn_df["pre_pt_root_id"],
    "postsynaptic_cell": syn_df["post_pt_root_id"],
    "dataset_id": DATASET_ID,
    "project_id": PROJECT_ID,
})
long_table = attach_linkml_metadata(
    pa.Table.from_pandas(long_df, schema=build_arrow_schema(SynapseConnectivityLong),
                         preserve_index=False),
    linkml_class="SynapseConnectivityLong",
)
write_deltalake(
    str(OUTPUT_ROOT / "synapse"),
    long_table,
    mode="overwrite",
    predicate=f"project_id = '{PROJECT_ID}' AND dataset_id = '{DATASET_ID}'",
    partition_by=["project_id"],
)
print("long rows written:", long_table.num_rows)

long rows written: 8204497


## Write 2 — wide per-synapse feature table → `synapsefeatures/<id>/`

Built from raw dataframes (not model instances), keyed by the synapse `id`,
with `synaptictargetlabel` LEFT-joined from the label feather. This mirrors how
`cellfeatures/` wide tables are handled — outside the model registry.

In [5]:
# Features are every syn_df column that is not the synapse id or a
# pre/post root id (those define the connection, not a feature).
feature_cols = [c for c in syn_df.columns if c != "id" and not c.endswith("_root_id")]
wide = syn_df[["id"] + feature_cols].merge(
    label_df[["id", "synaptictargetlabel"]], on="id", how="left",
)
wide["project_id"] = PROJECT_ID
wide["dataset_id"] = DATASET_ID
write_deltalake(
    str(OUTPUT_ROOT / "synapsefeatures" / FEATURE_MATRIX_ID),
    pa.Table.from_pandas(wide, preserve_index=False),
    mode="overwrite",
    predicate=f"project_id = '{PROJECT_ID}'",
    partition_by=["project_id"],
)
print("feature rows written:", len(wide),
      "| labelled:", int(wide["synaptictargetlabel"].notna().sum()))

feature rows written: 8204497 | labelled: 6706286


## Write 3 — `SynapseFeatureMatrix` pointer row → `synapsefeaturematrix/`

A single metadata row locating the wide feature table and naming its synapse-id
column, written through `write_models` (`(project_id, id)`-scoped per its
`WriteSpec`).

In [6]:
feature_matrix = SynapseFeatureMatrix(
    id=FEATURE_MATRIX_ID,
    description="Per-synapse position, size and target label for V1DD 1196.",
    dataset_id=DATASET_ID,
    project_id=PROJECT_ID,
    parquet_path=f"file://{(OUTPUT_ROOT / 'synapsefeatures' / FEATURE_MATRIX_ID).resolve()}/",
    synapse_index_column="id",
)
print("pointer rows written:", write_models([feature_matrix], output_root=OUTPUT_ROOT).rows_written)

pointer rows written: 1


## Verify

Read the long table back and LEFT-join the wide features. The join preserves
every synapse; unlabeled synapses get a null `synaptictargetlabel`.

In [7]:
full = read_synapse_table(
    PROJECT_ID, dataset_id=DATASET_ID,
    features=True, feature_matrix_id=FEATURE_MATRIX_ID,
    output_root=OUTPUT_ROOT,
)

In [8]:
full.head(3)

project_id,id,presynaptic_cell,postsynaptic_cell,dataset_id,pre_pt_position_x,pre_pt_position_y,pre_pt_position_z,post_pt_position_x,post_pt_position_y,post_pt_position_z,ctr_pt_position_x,ctr_pt_position_y,ctr_pt_position_z,size,synaptictargetlabel
str,str,str,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,i64,str
"""v1dd""","""354386968""","""864691132536286810""","""864691132734919083""","""v1dd_1196_em""",758200.5,802316.1,304380.0,757861.0,802558.6,304650.0,757967.7,802597.4,304380.0,240,"""shaft"""
"""v1dd""","""378070488""","""864691132572190492""","""864691132606767301""","""v1dd_1196_em""",792063.2,514342.5,183735.0,792664.6,514284.3,183915.0,792412.4,514294.0,183735.0,3056,"""shaft"""
"""v1dd""","""499493001""","""864691132573738810""","""864691132747578447""","""v1dd_1196_em""",977071.3,390075.8,191340.0,976974.3,390104.9,190935.0,976838.5,390337.7,190935.0,1346,null


In [9]:
full.select(["id", "presynaptic_cell", "postsynaptic_cell", "size", "synaptictargetlabel"]).head(3)

id,presynaptic_cell,postsynaptic_cell,size,synaptictargetlabel
str,str,str,i64,str
"""354386968""","""864691132536286810""","""864691132734919083""",240,"""shaft"""
"""378070488""","""864691132572190492""","""864691132606767301""",3056,"""shaft"""
"""499493001""","""864691132573738810""","""864691132747578447""",1346,null


---

## Cell–cell connectivity from synapses → `cellcellconnectivitylong_proofread_axon_to_dendrite/`

Aggregate the per-synapse `synapse/` delta table written above into
`CellCellConnectivityLong` pairs, restricted to the **proofread axons cohort**
(`v1dd_1196_proofread_axons`) as presynaptic and the **proofread dendrites
cohort** (`v1dd_1196_proofread_dendrites`) as postsynaptic — the two DataSets
registered by `etl_v1dd_02_cave.ipynb`. Two measurement types are emitted per
pair: `SYNAPSE_COUNT` (number of synapses) and `SUM_ANATOMICAL_SIZE` (total
synapse `size`).

In [10]:
import polars as pl

from connects_common_connectivity.models import (
    CellCellConnectivityLong,
    Modality,
    SynapticMeasurementType,
    Unit,
)
from connects_common_connectivity.io.arrow_utils import models_to_table

DATASET_PROOFREAD_AXON = "v1dd_1196_proofread_axons"
DATASET_PROOFREAD_DEND = "v1dd_1196_proofread_dendrites"
CONN_TABLE = "cellcellconnectivitylong_proofread_axon_to_dendrite"

# Cohort membership (pre = proofread axons, post = proofread dendrites) comes from
# the DataItemDataSetAssociation delta written by etl_v1dd_02_cave.ipynb.
assoc = pl.read_delta(str(OUTPUT_ROOT / "dataitem_dataset_association")).filter(
    pl.col("project_id") == PROJECT_ID
)
axon_ids = set(
    assoc.filter(pl.col("dataset_id") == DATASET_PROOFREAD_AXON)["dataitem_id"].to_list()
)
dend_ids = set(
    assoc.filter(pl.col("dataset_id") == DATASET_PROOFREAD_DEND)["dataitem_id"].to_list()
)
print(f"proofread axon cells (pre): {len(axon_ids)} | "
      f"proofread dendrite cells (post): {len(dend_ids)}")

proofread axon cells (pre): 1164 | proofread dendrite cells (post): 63986


Read the per-synapse table back from the delta lake (with the `size` feature
joined in), keep only synapses whose presynaptic cell is a proofread axon and
whose postsynaptic cell is a proofread dendrite, then aggregate to one row per
`(pre, post)` pair.

In [11]:
syn = read_synapse_table(
    PROJECT_ID, dataset_id=DATASET_ID,
    features=True, feature_matrix_id=FEATURE_MATRIX_ID,
    output_root=OUTPUT_ROOT,
)

pairs = (
    syn.filter(
        pl.col("presynaptic_cell").is_in(axon_ids)
        & pl.col("postsynaptic_cell").is_in(dend_ids)
    )
    .group_by(["presynaptic_cell", "postsynaptic_cell"])
    .agg(
        pl.len().alias("n_syn"),
        pl.col("size").sum().alias("sum_size"),
    )
)
print(f"connected (axon, dendrite) pairs: {pairs.shape[0]} "
      f"(from {int(pairs['n_syn'].sum())} synapses)")

connected (axon, dendrite) pairs: 739022 (from 1431746 synapses)


Build two `CellCellConnectivityLong` rows per pair — `SYNAPSE_COUNT` (unit
`COUNT`) and `SUM_ANATOMICAL_SIZE` (unit `ARBITRARY_UNIT`) — and write them with
a `project_id`-scoped overwrite, partitioned by `project_id` and
`measurement_type` (matching the Minnie65 cell-cell layout).

In [12]:
rows = []
for row in pairs.iter_rows(named=True):
    pre, post = row["presynaptic_cell"], row["postsynaptic_cell"]
    rows.append(
        CellCellConnectivityLong(
            id=f"{pre}_{post}_{SynapticMeasurementType.SYNAPSE_COUNT.value}",
            presynaptic_cell=pre,
            postsynaptic_cell=post,
            measurement_type=SynapticMeasurementType.SYNAPSE_COUNT.value,
            modality=Modality.ELECTRON_MICROSCOPY.value,
            value=float(row["n_syn"]),
            unit=Unit.COUNT.value,
            project_id=PROJECT_ID,
        )
    )
    rows.append(
        CellCellConnectivityLong(
            id=f"{pre}_{post}_{SynapticMeasurementType.SUM_ANATOMICAL_SIZE.value}",
            presynaptic_cell=pre,
            postsynaptic_cell=post,
            measurement_type=SynapticMeasurementType.SUM_ANATOMICAL_SIZE.value,
            modality=Modality.ELECTRON_MICROSCOPY.value,
            value=float(row["sum_size"]),
            unit=Unit.ARBITRARY_UNIT.value,
            project_id=PROJECT_ID,
        )
    )

conn_table = attach_linkml_metadata(
    models_to_table(rows, schema=build_arrow_schema(CellCellConnectivityLong)),
    linkml_class="CellCellConnectivityLong",
)
write_deltalake(
    str(OUTPUT_ROOT / CONN_TABLE),
    conn_table,
    mode="overwrite",
    predicate=f"project_id = '{PROJECT_ID}'",
    partition_by=["project_id", "measurement_type"],
)
print(f"CellCellConnectivityLong rows written: {conn_table.num_rows} "
      f"({len(rows) // 2} pairs x 2 measurement types)")

CellCellConnectivityLong rows written: 1478044 (739022 pairs x 2 measurement types)


### Verify

Read the connectivity table back and confirm both measurement types are present
and every row survives the round-trip.

In [13]:
conn_v = pl.read_delta(str(OUTPUT_ROOT / CONN_TABLE)).filter(
    pl.col("project_id") == PROJECT_ID
)
print("shape:", conn_v.shape)
print(conn_v.group_by("measurement_type").len())
assert conn_v.shape[0] == len(rows)
assert conn_v["measurement_type"].n_unique() == 2
conn_v.head(3)

shape: (1478044, 9)
shape: (2, 2)
┌─────────────────────┬────────┐
│ measurement_type    ┆ len    │
│ ---                 ┆ ---    │
│ str                 ┆ u32    │
╞═════════════════════╪════════╡
│ SYNAPSE_COUNT       ┆ 739022 │
│ SUM_ANATOMICAL_SIZE ┆ 739022 │
└─────────────────────┴────────┘


project_id,id,description,presynaptic_cell,postsynaptic_cell,measurement_type,modality,value,unit
str,str,str,str,str,str,str,f64,str
"""v1dd""","""864691132657090520_86469113276…",null,"""864691132657090520""","""864691132760724359""","""SYNAPSE_COUNT""","""ELECTRON_MICROSCOPY""",1.0,"""COUNT"""
"""v1dd""","""864691132635522574_86469113267…",null,"""864691132635522574""","""864691132671640090""","""SYNAPSE_COUNT""","""ELECTRON_MICROSCOPY""",1.0,"""COUNT"""
"""v1dd""","""864691132687686538_86469113282…",null,"""864691132687686538""","""864691132827947240""","""SYNAPSE_COUNT""","""ELECTRON_MICROSCOPY""",8.0,"""COUNT"""
